In [12]:
import numpy as np
import pandas as pd

def creer_plateau(R_case, R_dragon, R_tresor):
    plateau = {}

    for ligne in range(4):
        for colonne in range(4):
            plateau[(ligne, colonne)] = ("vide", R_case)

    plateau[(0,0)] = ("Depart", R_case)

    case_dragons = [(0, 1), (2, 1), (3, 2), (1, 3)]

    for position in case_dragons:
        plateau[position] = ("Dragon", R_dragon)

    plateau[(3, 3)] = ("Tresor", R_tresor)

    return plateau

In [13]:
plateau = creer_plateau(0, -1, 1)

print(plateau[(0, 0)])
print(plateau[(1, 2)])
print(plateau[(3, 3)])

('Depart', 0)
('vide', 0)
('Tresor', 1)


In [14]:
def application_action(action, position, plateau):
    x, y = position
    match action:
        case 0:
            y -= 1
        case 1:
            x += 1
        case 2:
            y += 1
        case 3:
            x -= 1
        case _:
            print("probleme")

    new_pos = (x,y)   

    if new_pos not in plateau:
        new_pos = position
        reward = 0
        fin = False
        return new_pos, reward, fin

    type_case, reward = plateau[new_pos]

    fin = type_case == "Dragon" or type_case == "Tresor"

    return new_pos, reward, fin 
    

In [20]:
plateau = creer_plateau(0, -1, 1)

print("Déplacement simple :", application_action(1, (0, 0), plateau))
print("Sortie du plateau :", application_action(0, (0, 0), plateau))
print("Dragon :", application_action(2, (0, 0), plateau))
print("Trésor :", application_action(1, (2, 3), plateau))

Déplacement simple : ((1, 0), 0, False)
Sortie du plateau : ((0, 0), 0, False)
Dragon : ((0, 1), -1, True)
Trésor : ((3, 3), 1, True)


In [18]:
print("Dragon :", application_action(2, (0, 0), plateau))

Dragon : ((0, 1), -1, True)


# PARTIE 2 :

In [25]:
def choose_action(state, epsilon, mat_q):
    if np.random.uniform(0, 1) < epsilon:
        return np.random.randint(4)
    sortie_q = mat_q[state]
    meilleures_actions = np.flatnonzero(sortie_q == np.max(sortie_q))

    return int(np.random.choice(meilleures_actions))


In [26]:
mat_q = np.zeros((16, 4))
mat_q[5] = [0.2, 0.8, -0.5, 0.8]

print("Exploration :", [choose_action(5, 1, mat_q) for _ in range(20)])
print("Exploitation :", [choose_action(5, 0, mat_q) for _ in range(20)])

Exploration : [0, 2, 0, 2, 1, 0, 3, 3, 2, 2, 1, 2, 2, 0, 2, 3, 2, 2, 2, 3]
Exploitation : [1, 3, 3, 1, 1, 3, 3, 3, 1, 3, 1, 3, 3, 1, 1, 1, 3, 3, 3, 1]


In [2]:
def onestep(mat_q, state, epsilon, alpha, gamma, plateau):
    """
    Joue un coup et met à jour la table Q.

    mat_q   : table Q (16 x 4), modifiée sur place
    state   : état courant (0..15), state = 4 * ligne + col
    epsilon : probabilité d'explorer (action au hasard)
    alpha   : pas d'apprentissage (fraction de l'erreur corrigée)
    gamma   : facteur d'actualisation (poids des gains futurs)
    plateau : dictionnaire créé par creer_plateau

    Renvoie : mat_q, new_state, reward, fin
    """
    # 1. Choix de l'action (ε-greedy) : explorer ou exploiter la table Q
    action = choose_action(state, epsilon, mat_q)

    # 2. Conversion état -> position (ligne, col), puis application de l'action
    #    ex. state = 6 -> divmod(6, 4) = (1, 2)
    position = divmod(state, 4)
    new_position, reward, fin = application_action(action, position, plateau)

    # 3. Conversion de la nouvelle position -> numéro d'état
    #    ex. (1, 2) -> 4*1 + 2 = 6
    ligne, col = new_position
    new_state = 4 * ligne + col

    # 4. Cible (équation de Bellman) :
    #    gain immédiat + valeur actualisée de la meilleure action depuis la nouvelle case.
    #    Si la partie est finie (dragon ou trésor), il n'y a pas de suite : (1 - fin) = 0
    cible = reward + gamma * (1 - fin) * np.max(mat_q[new_state])

    # 5. Mise à jour : on rapproche Q(s, a) de la cible d'une fraction alpha de l'erreur
    erreur = cible - mat_q[state, action]
    mat_q[state, action] = mat_q[state, action] + alpha * erreur

    return mat_q, new_state, reward, fin